# 02 Label Review

Use this notebook to inspect selected artworks and review manual labels.

In [ ]:
from pathlib import Path
import random
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

LABELS_PATH = ROOT / 'data' / 'labels' / 'critique_labels_v1.csv'
labels = pd.read_csv(LABELS_PATH).fillna('')
print(labels.shape)
labels.head()

## Show One Artwork

Change `row_index` to inspect a specific row.

In [ ]:
row_index = 0
row = labels.iloc[row_index]
image_path = ROOT / row['image_path']

print(row[['id', 'artist', 'style', 'genre', 'composition', 'color_harmony', 'contrast', 'lighting', 'perspective', 'critique_notes']])

if image_path.exists():
    img = Image.open(image_path)
    plt.figure(figsize=(7, 7))
    plt.imshow(img)
    plt.axis('off')
else:
    print('Image not found:', image_path)

## Random Review Grid

Use this to spot label mistakes quickly.

In [ ]:
sample_size = min(9, len(labels))
sample = labels.sample(sample_size, random_state=random.randint(1, 9999)).reset_index(drop=True)

fig, axes = plt.subplots(3, 3, figsize=(12, 12))
axes = axes.flatten()

for ax, (_, row) in zip(axes, sample.iterrows()):
    image_path = ROOT / row['image_path']
    ax.axis('off')
    title = f"{row['id']}\ncomp={row['composition']} color={row['color_harmony']}\ncontrast={row['contrast']} light={row['lighting']}"
    ax.set_title(title, fontsize=9)
    if image_path.exists():
        ax.imshow(Image.open(image_path))
    else:
        ax.text(0.5, 0.5, 'missing image', ha='center', va='center')

for ax in axes[len(sample):]:
    ax.axis('off')

plt.tight_layout()

## Missing Label Rows

These rows still need manual labeling.

In [ ]:
label_cols = ['composition', 'color_harmony', 'contrast', 'lighting', 'perspective']
missing_mask = labels[label_cols].eq('').any(axis=1)
missing = labels.loc[missing_mask, ['id', 'image_path', 'artist', 'genre', *label_cols]]
print('Missing rows:', len(missing))
missing.head(20)